# Scaling ML Systems — AI Lab Instructor Notebook

*MLOps & Systems · Expert*

Simulate serving load and study how queueing, micro-batching, and caching change latency and throughput.

**Outcome.** Students can reason about p95/p99 latency, throughput, batching windows, cache hit rates, and scaling tradeoffs in ML serving systems.

3 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
from collections import deque

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

## Section 0 — Workload (Poisson arrivals)

### Task 0.1
Generate an arrival stream for a loaded serving system.

- use exponential inter-arrival times drawn from an exponential distribution to simulate Poisson arrivals
- do not confuse this with exponential growth over time; these are random gaps between requests
- accumulate them into monotonic arrival timestamps in seconds
- create random query vectors so later sections have a synthetic request payload

This section defines the workload that the serving simulator will process.

In [ ]:
N_REQ = 2000
ARRIVAL_RATE = 220.0  # req/s
D = 128

def poisson_arrivals(n: int, rate: float, seed: int = 0):
    rng = np.random.default_rng(seed)
    inter = rng.exponential(1.0 / rate, size=(n,))
    return np.cumsum(inter)

arrivals_s = poisson_arrivals(N_REQ, ARRIVAL_RATE, seed=0)
queries = rng.standard_normal((N_REQ, D)).astype(np.float64)
check('monotonic_arrivals', np.all(np.diff(arrivals_s) >= 0.0))


In [ ]:
# Checks
check('monotonic_arrivals', np.all(np.diff(arrivals_s) >= 0.0))

**Why this works.** # Scaling ML Systems — Instructor Lab

Offline lab: simulate serving load to study latency/throughput tradeoffs (queueing, batching, caching).

## 0 — Workload (Poisson arrivals)

## 1 — Simulator (single worker, micro-batching, caching)

## Task 2: 1 — Simulator (single worker, micro-batching, caching)

## Section 1 — Simulator (single worker, micro-batching, caching)

### Task 1.1
Implement a simple single-worker serving simulator.

- `service_time_ms(batch_size)`: batch service-time model
- `simulate(...)`: process requests in arrival order
- support `max_batch` and `max_wait_ms` for micro-batching
- support `cache_hit_rate` so some requests bypass expensive inference
- return latency percentiles, throughput, cache-hit count, and mean batch size

Then run a no-batching, no-cache baseline and inspect its latency profile.

In [ ]:
$f

In [ ]:
# Checks
check('p95_ge_p50', baseline['p95_ms'] >= baseline['p50_ms'])
check('thr_positive', baseline['throughput_rps'] > 0.0)
check('cache_count_nonneg', baseline['n_cache_hits'] >= 0)


## Task 3: 2 — Experiments: batching vs caching

## Section 2 — Experiments: batching vs caching

### Task 2.1
Compare several serving configurations on the same arrival stream.

- no batching / no cache baseline
- batching only
- caching only
- batching plus caching

Report p95 latency, p99 latency, and throughput for each setting, then check that scaling knobs actually change the system behavior.

In [ ]:
$10

In [ ]:
# Checks
check('got_results', len(results) >= 3)
check('batching_or_cache_helps_thr', max(r['throughput_rps'] for r in results) >= results[0]['throughput_rps'])
check('cache_not_worse_than_baseline', results_by_name['no_batch_cache30']['p95_ms'] <= results_by_name['no_batch_no_cache']['p95_ms'])
